# AudioGen vs MusicGen — Colab Demo

**Prompt**: "lo-fi hiphop dark" で30秒生成。両モデルを聴き比べ。

Meta AudioCraft: https://github.com/facebookresearch/audiocraft

In [ ]:
# 1. Install
!pip install -U audiocraft torchaudio

In [ ]:
# 2. Check GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import time
import torchaudio
from audiocraft.models import AudioGen, MusicGen
from audiocraft.data.audio import audio_write

PROMPT = "lo-fi hiphop dark"
DURATION = 30  # seconds

In [ ]:
# 3a. AudioGen — 環境音生成モデル
print("Loading AudioGen (medium)...")
t0 = time.time()
audiogen = AudioGen.get_pretrained("facebook/audiogen-medium")
audiogen.set_generation_params(duration=DURATION)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Generate with AudioGen
print(f"Generating '{PROMPT}' with AudioGen ({DURATION}s)...")
t0 = time.time()
wav_ag = audiogen.generate([PROMPT], progress=True)
print(f"Generated in {time.time()-t0:.1f}s")
print(f"Shape: {wav_ag.shape}")

# Save
audio_write("audiogen_output", wav_ag[0].cpu(), audiogen.sample_rate,
            strategy="loudness", loudness_compressor=True)
print("Saved: audiogen_output.wav")

In [ ]:
# 3b. MusicGen — 音楽生成モデル
print("Loading MusicGen (medium)...")
t0 = time.time()
musicgen = MusicGen.get_pretrained("facebook/musicgen-medium")
musicgen.set_generation_params(duration=DURATION)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Generate with MusicGen
print(f"Generating '{PROMPT}' with MusicGen ({DURATION}s)...")
t0 = time.time()
wav_mg = musicgen.generate([PROMPT], progress=True)
print(f"Generated in {time.time()-t0:.1f}s")
print(f"Shape: {wav_mg.shape}")

# Save
audio_write("musicgen_output", wav_mg[0].cpu(), musicgen.sample_rate,
            strategy="loudness", loudness_compressor=True)
print("Saved: musicgen_output.wav")

In [ ]:
# 4. Playback
from IPython.display import Audio, display

print("=== AudioGen ===")
display(Audio(wav_ag[0].cpu().numpy(), rate=audiogen.sample_rate))

print("=== MusicGen ===")
display(Audio(wav_mg[0].cpu().numpy(), rate=musicgen.sample_rate))

In [ ]:
# 5. Download to local
from google.colab import files
files.download("audiogen_output.wav")
files.download("musicgen_output.wav")